## Using PyTorch

In [1]:
import torch
import torch.nn as nn

In [13]:
class NeuralNetwork:
    def __init__(self, lr = 0.01, epochs=100):
        self.lr = lr
        self.epochs = epochs
        self.model = None

    def build_model(self, num_neurons = 10):
        self.model = nn.Sequential(
                # Hidden Layers
                nn.Linear(2, num_neurons),
                nn.ReLU(),
                nn.Linear(num_neurons, num_neurons),
                nn.ReLU(),
                nn.Linear(num_neurons, num_neurons),
                nn.ReLU(),
                # O/P layer
                nn.Linear(num_neurons, 1),           
                nn.Sigmoid()
        )
        # return self

    def fit(self, X, y):
        optimiser = torch.optim.Adam(self.model.parameters(),lr = self.lr)
        criterion = nn.BCELoss()
        for epoch in range(self.epochs):
            for input, target in zip(X, y):
                input = torch.tensor(input, dtype=torch.float32).unsqueeze(0)
                target = torch.tensor([[target]], dtype=torch.float32)

                # Forward Pass
                y_hat = self.model(input)

                # Loss calculation
                loss = criterion(y_hat, target)

                # Clear the old gradients
                optimiser.zero_grad()

                # Back propagation
                loss.backward()

                # Weight updates
                optimiser.step()
        # return self

    def predict(self, X):
        return self.model(torch.tensor(X, dtype=torch.float32).unsqueeze(0))

In [16]:
model = NeuralNetwork(0.01, 100)
model.build_model()
X = [(0,0), (0,1), (1,0), (1,1)]
y = [0, 1, 1, 0]
model.fit(X,y)

In [19]:
model.predict(X[0])

tensor([[0.0008]], grad_fn=<SigmoidBackward0>)

## Without using PyTorch

In [1]:
import numpy as np

class NeuralNetwork:

    def __init__(self, num_neurons=10, lr=0.05, epochs=20000):

        self.num_neurons = num_neurons
        self.lr = lr
        self.epochs = epochs

        # Initialize weights and biases
        self.W1 = np.random.randn(2, num_neurons) * np.sqrt(2 / 2)
        self.b1 = np.zeros((1, num_neurons))

        self.W2 = np.random.randn(num_neurons, num_neurons) * np.sqrt(2 / num_neurons)
        self.b2 = np.zeros((1, num_neurons))

        self.W3 = np.random.randn(num_neurons, num_neurons) * np.sqrt(2 / num_neurons)
        self.b3 = np.zeros((1, num_neurons))

        self.W4 = np.random.randn(num_neurons, 1) * np.sqrt(2 / num_neurons)
        self.b4 = np.zeros((1, 1))


    def relu(self, Z):
        return np.maximum(0, Z)


    def relu_derivative(self, Z):
        return (Z > 0).astype(float)


    def sigmoid(self, Z):
        Z = np.clip(Z, -500, 500)
        return 1 / (1 + np.exp(-Z))


    def forward(self, X):

        # First hidden layer
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.relu(self.Z1)

        # Second hidden layer
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.relu(self.Z2)

        # Third hidden layer
        self.Z3 = np.dot(self.A2, self.W3) + self.b3
        self.A3 = self.relu(self.Z3)

        # Output layer
        self.Z4 = np.dot(self.A3, self.W4) + self.b4
        self.Y_hat = self.sigmoid(self.Z4)

        return self.Y_hat


    def calculate_loss(self, Y, Y_hat):

        epsilon = 1e-10
        Y_hat = np.clip(Y_hat, epsilon, 1 - epsilon)

        loss = -np.mean(
            Y * np.log(Y_hat) +
            (1 - Y) * np.log(1 - Y_hat)
        )

        return loss


    def backward(self, X, Y):

        m = X.shape[0]

        # Output layer
        dZ4 = (self.Y_hat - Y) / m
        dW4 = np.dot(self.A3.T, dZ4)
        db4 = np.sum(dZ4, axis=0, keepdims=True)

        # Hidden layer 3
        dA3 = np.dot(dZ4, self.W4.T)
        dZ3 = dA3 * self.relu_derivative(self.Z3)
        dW3 = np.dot(self.A2.T, dZ3)
        db3 = np.sum(dZ3, axis=0, keepdims=True)

        # Hidden layer 2
        dA2 = np.dot(dZ3, self.W3.T)
        dZ2 = dA2 * self.relu_derivative(self.Z2)
        dW2 = np.dot(self.A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        # Hidden layer 1
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.relu_derivative(self.Z1)
        dW1 = np.dot(X.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        return dW1, db1, dW2, db2, dW3, db3, dW4, db4


    def update_weights(self, gradients):

        dW1, db1, dW2, db2, dW3, db3, dW4, db4 = gradients

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3

        self.W4 -= self.lr * dW4
        self.b4 -= self.lr * db4


    def fit(self, X, Y, verbose=False):

        for epoch in range(self.epochs):

            # Forward pass
            Y_hat = self.forward(X)

            # Calculate loss
            loss = self.calculate_loss(Y, Y_hat)

            # Backpropagation
            gradients = self.backward(X, Y)

            # Update weights
            self.update_weights(gradients)

            if verbose and (epoch + 1) % 1000 == 0:
                print(f"Epoch {epoch + 1}: Loss = {loss:.6f}")

        return self


    def predict(self, X):

        probabilities = self.forward(X)
        return (probabilities >= 0.5).astype(int)

In [2]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

boolean_functions = {
    "FALSE": np.array([[0], [0], [0], [0]]),
    "AND": np.array([[0], [0], [0], [1]]),
    "x1 AND NOT x2": np.array([[0], [0], [1], [0]]),
    "x1": np.array([[0], [0], [1], [1]]),
    "NOT x1 AND x2": np.array([[0], [1], [0], [0]]),
    "x2": np.array([[0], [1], [0], [1]]),
    "XOR": np.array([[0], [1], [1], [0]]),
    "OR": np.array([[0], [1], [1], [1]]),
    "NOR": np.array([[1], [0], [0], [0]]),
    "XNOR": np.array([[1], [0], [0], [1]]),
    "NOT x2": np.array([[1], [0], [1], [0]]),
    "x1 OR NOT x2": np.array([[1], [0], [1], [1]]),
    "NOT x1": np.array([[1], [1], [0], [0]]),
    "NOT x1 OR x2": np.array([[1], [1], [0], [1]]),
    "NAND": np.array([[1], [1], [1], [0]]),
    "TRUE": np.array([[1], [1], [1], [1]])
}

In [7]:
results = {}

for name, Y in boolean_functions.items():

    print("\n" + "=" * 60)
    print(f"Training: {name}")
    print("=" * 60)

    # Create a fresh neural network
    model = NeuralNetwork(
        num_neurons=10,
        lr=0.05,
        epochs=20000
    )
    # Train the network
    model.fit(
        X,
        Y,
        verbose=False
    )

    # Get probabilities
    probabilities = model.predict(X)

    # Convert probabilities into 0 or 1
    predictions = model.predict(X)

    # Store results
    results[name] = {
        "target": Y.flatten(),
        "probabilities": probabilities.flatten(),
        "predictions": predictions.flatten()
    }

    # Display results
    print("\nInput | Target | Probability | Prediction")
    print("-" * 45)

    for x, target, probability, prediction in zip(
        X,
        Y.flatten(),
        probabilities.flatten(),
        predictions.flatten()
    ):

        print(
            f"{x.astype(int)}  |"
            f"   {target}    |"
            f"   {probability:.4f}    |"
            f"     {prediction}"
        )


Training: FALSE

Input | Target | Probability | Prediction
---------------------------------------------
[0 0]  |   0    |   0.0000    |     0
[0 1]  |   0    |   0.0000    |     0
[1 0]  |   0    |   0.0000    |     0
[1 1]  |   0    |   0.0000    |     0

Training: AND

Input | Target | Probability | Prediction
---------------------------------------------
[0 0]  |   0    |   0.0000    |     0
[0 1]  |   0    |   0.0000    |     0
[1 0]  |   0    |   0.0000    |     0
[1 1]  |   1    |   1.0000    |     1

Training: x1 AND NOT x2

Input | Target | Probability | Prediction
---------------------------------------------
[0 0]  |   0    |   0.0000    |     0
[0 1]  |   0    |   0.0000    |     0
[1 0]  |   1    |   1.0000    |     1
[1 1]  |   0    |   0.0000    |     0

Training: x1

Input | Target | Probability | Prediction
---------------------------------------------
[0 0]  |   0    |   0.0000    |     0
[0 1]  |   0    |   0.0000    |     0
[1 0]  |   1    |   1.0000    |     1
[1 

In [5]:
print("\n")
print("=" * 65)
print("FINAL RESULTS FOR ALL 16 BOOLEAN FUNCTIONS")
print("=" * 65)


for name, result in results.items():

    target = list(result["target"])
    prediction = list(result["predictions"])

    correct = target == prediction

    print(
        f"{name:<22}"
        f"Target: {target}"
        f" | Predicted: {prediction}"
        f" | Correct: {correct}"
    )



FINAL RESULTS FOR ALL 16 BOOLEAN FUNCTIONS
FALSE                 Target: [np.int64(0), np.int64(0), np.int64(0), np.int64(0)] | Predicted: [np.int64(0), np.int64(0), np.int64(0), np.int64(0)] | Correct: True
AND                   Target: [np.int64(0), np.int64(0), np.int64(0), np.int64(1)] | Predicted: [np.int64(0), np.int64(0), np.int64(0), np.int64(1)] | Correct: True
x1 AND NOT x2         Target: [np.int64(0), np.int64(0), np.int64(1), np.int64(0)] | Predicted: [np.int64(0), np.int64(0), np.int64(1), np.int64(0)] | Correct: True
x1                    Target: [np.int64(0), np.int64(0), np.int64(1), np.int64(1)] | Predicted: [np.int64(0), np.int64(0), np.int64(1), np.int64(1)] | Correct: True
NOT x1 AND x2         Target: [np.int64(0), np.int64(1), np.int64(0), np.int64(0)] | Predicted: [np.int64(0), np.int64(1), np.int64(0), np.int64(0)] | Correct: True
x2                    Target: [np.int64(0), np.int64(1), np.int64(0), np.int64(1)] | Predicted: [np.int64(0), np.int64(1), np.int6